# 46 — Stage-B Medium Confirmatory Pilot
**Project:** Causal-History Dependence in Robot Policy Recovery  
**Phase:** 11 (Benchmark-Readiness Audit and Staged Pilot-Scale Confirmation)  
**Status:** `Confirmatory pilot — exploratory validation`  

### Objectives:
1. Audit Stage B confirmatory pilot (2,400 rollouts: 80 scenarios x 4 policies x 5 conditions x 3 seeds).
2. Quantify empirical discordance, continuous trajectory divergence, policy rankings ($	au$), and safety.
3. Validate Go/No-Go Gate 2 for benchmark readiness.

In [1]:
import os, sys
cwd = os.getcwd()
if os.path.exists(os.path.join(cwd, "docs")):
    repo_root = cwd
elif os.path.exists(os.path.join(cwd, "..", "docs")):
    repo_root = os.path.abspath(os.path.join(cwd, ".."))
elif os.path.exists(os.path.join(cwd, "causal-history-recovery", "docs")):
    repo_root = os.path.abspath(os.path.join(cwd, "causal-history-recovery"))
else:
    repo_root = os.path.abspath(os.path.join(cwd, ".."))

src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json

stage_b_dir = os.path.join(repo_root, "results", "phase_11", "stage_b_pilot")
with open(os.path.join(stage_b_dir, "evaluation_summary.json")) as f:
    b_eval = json.load(f)

with open(os.path.join(stage_b_dir, "policy_ranking_report.json")) as f:
    b_rank = json.load(f)

with open(os.path.join(stage_b_dir, "history_dependence_report.json")) as f:
    b_hr = json.load(f)

print("Stage B Total Rollouts:", b_eval["total_rollouts"])
print("Policy Ranking Stability (Kendall's tau):", b_rank["kendall_tau"])


Stage B Total Rollouts: 3720
Policy Ranking Stability (Kendall's tau): -0.5163977794943223


### 2. Paired History Dependence and Distance Divergence (Stage B)

In [2]:

print(f"{'Policy':<25} {'H_R':<8} {'R(B0)':<10} {'R(B4)':<10} {'Dist B0 (m)':<15} {'Dist B4 (m)':<15} {'Dist Div (m)':<15}")
print("-" * 98)
for p, data in b_hr["policies"].items():
    print(f"{p:<25} {data['H_R']:<8.3f} {data['R_B0']:<10.3f} {data['R_B4']:<10.3f} {data['mean_final_distance_B0']:<15.4f} {data['mean_final_distance_B4']:<15.4f} {data['distance_divergence_B0_B4']:<15.4f}")


Policy                    H_R      R(B0)      R(B4)      Dist B0 (m)     Dist B4 (m)     Dist Div (m)   
--------------------------------------------------------------------------------------------------
reactive_mlp              0.032    0.032      0.000      0.2491          0.2524          0.0033         
history_gru               0.000    0.000      0.000      0.2449          0.2449          0.0000         
recovery_augmented_mlp    0.016    0.016      0.000      0.2646          0.2641          0.0006         
recovery_augmented_gru    -0.016   0.000      0.016      0.2641          0.2674          0.0033         


### 3. Sensitivity Breakdown Across Tasks and Failure Modes

In [3]:

with open(os.path.join(stage_b_dir, "sensitivity_report.json")) as f:
    sens_b = json.load(f)

print("Sensitivity by Task:")
for t, stats in sens_b["by_task"].items():
    print(f"  {t}: Success Rate = {stats['success_rate']*100:.2f}%, Mean Dist = {stats['mean_distance']:.4f} m")

print("\nSensitivity by Failure Mode:")
for f_mode, stats in sens_b["by_failure"].items():
    print(f"  {f_mode}: Success Rate = {stats['success_rate']*100:.2f}%, Mean Dist = {stats['mean_distance']:.4f} m")


Sensitivity by Task:
  task1_cube: Success Rate = 0.88%, Mean Dist = 0.2455 m
  task2_puck: Success Rate = 0.23%, Mean Dist = 0.2784 m

Sensitivity by Failure Mode:
  rotational_slip: Success Rate = 0.00%, Mean Dist = 0.2591 m
  contact_loss: Success Rate = 1.00%, Mean Dist = 0.2562 m


### 4. Go/No-Go Gate 2 Verification
Confirm: sample completion >= 2400 rollouts, continuous metric sensitivity, valid finite distributions, and zero safety violations.

In [4]:

gate2_passed = (
    b_eval["total_rollouts"] >= 2400 and
    all(data["mean_final_distance_B0"] > 0 for data in b_hr["policies"].values()) and
    len(b_hr["policies"]) == 4
)
print("Go/No-Go Gate 2 Verdict:", "PASSED (GO)" if gate2_passed else "FAILED (NO-GO)")
assert gate2_passed is True


Go/No-Go Gate 2 Verdict: PASSED (GO)
